Q1.
(a) None
이유: re.match()는 문야열의 시작 부분부터 일치하느지 확인하기 때문이다.

(b): '2026-05-06'
이유: re.search()는 문자열 전체에서 패턴과 일치하는 첫 번째 문자열을 찾기 때문이다.

(c): ['2026-05-06', '2026-05-18']
이유: 캡처 그룹이 없는 상태에서 re.findall은 일치하는 모든 것을 찾아 리스트로 반환하기 때문이다.

(d) [('2026', '05', '06'), ('2026', '05', '18')]
이유: 3개의 캡처 그룹이 존재하기 때문에 re.findall은 튜플 형태로 구성된 리스트를 반환한다.

(e) ['2026-05-06', '2026-05-18']
이유: ? 뒤에 오는 것은 비캡처그룹이기 때문에 (c)와 동일하다.

※ 추가
re.findall은 패턴에 괄호를 사용한 캡처 그룹이 있으면 전체 매칭 문자열 대신 그룹에 매칭된 부분만 튜플 형태로 추출하여 반환한다.
이와 달리 괄호가 없거나 (?:) 형태의 비캡처 그룹을 사용하면 그룹을 무시하고 패턴 전체와 일치하는 문자열을 리스트로 반환하기 때문이다.

Q2.
(a) '[T]!'
이유: .+는 기본적으로 탐욕적 수량자이기때문에 가능한 긴 범위를 찾는다.

(b) '[T]안녕[T] [T]세상[T]!'
이유: .+? 게으른 수량자이므로 가능한 짧은 범위를 찾는다.

(c) '[T]안녕[T] [T]세상[T]!'
이유: [^>]+는 >가 아닌 문자들을 의미하기 때문에 닫는 괄호를 만나면 매칭을 끊는다.

(d)'수강생 <30>명, 조교 <3>명'
이유: 원시 문자열을 이용하여 \1이 다시 그대로 전달된다.

(e)'수강생 <\x01>명, 조교 <\x01>명'
이유: 원시 문자열이 아닌 일반 문자열을 사용했기 때문에 \1을 8진수 탈출 문자로 받아들여 SOH로 바꿔버린다.

※ 추가 i
.+는 가능한 가장 긴 문자열을 매칭하는 탐욕적 속성,
.+?는 가능한 가장 짧은 문자열을 매칭하는 게으른 속성을 가지기 때문이다.

※ 추가 ii
d는 r을 이용해 \1이 정규표현식의 1번 캡쳐 그룹을 역참조하라고 해석된다.
e의 경우는 일반 문자열로 해석하기 때문에 \1을 아스키 제어문자로 먼저 해석한다.

In [4]:
import re
from collections import Counter

# 동일한 패턴의 반복 사용을 위해 re.compile로 미리 컴파일
URL_PATTERN = re.compile(r"https?://\S*")
HTML_PATTERN = re.compile(r"<[^>]+>")
EMAIL_PATTERN = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
PHONE_PATTERN = re.compile(r"\d{2,4}-\d{3,4}-\d{4}")
MENTION_HASHTAG_PATTERN = re.compile(r"[@#]\w+")
JAMO_PATTERN = re.compile(r"[ㄱ-ㅎㅏ-ㅣ]+")
WHITESPACE_PATTERN = re.compile(r"\s+")
HASHTAG_EXTRACT_PATTERN = re.compile(r"#(\w+)")


def clean_post(post: str) -> str:
    """한 건의 게시물을 지정된순서대로 정제하여 반환"""
    p = URL_PATTERN.sub(" ", post)
    
    p = HTML_PATTERN.sub("", p)
    
    p = EMAIL_PATTERN.sub("[이메일]", p)
    p = PHONE_PATTERN.sub("[전화]", p)
    
    p = MENTION_HASHTAG_PATTERN.sub(" ", p)
    
    p = JAMO_PATTERN.sub("", p)
    
    p = WHITESPACE_PATTERN.sub(" ", p).strip()
    
    return p


def extract_hashtags(post: str) -> list[str]:
    """원본 입력 문자열로부터 #을 제외한 해시태그 이름들을 리스트로 추출"""
    return HASHTAG_EXTRACT_PATTERN.findall(post)


def analyze_posts(posts: list[str]) -> dict:
    """게시물 리스트를 받아 정제 후 통계 정보를 담은 딕셔너리를 반환"""
    posts_n = len(posts)
    total_length = 0
    total_masked = 0
    all_hashtags = []
    
    for post in posts:
        cleaned = clean_post(post)
        total_length += len(cleaned)
        
        p_temp = URL_PATTERN.sub(" ", post)
        p_temp = HTML_PATTERN.sub("", p_temp)
        _, email_cnt = EMAIL_PATTERN.subn("[이메일]", p_temp)
        _, phone_cnt = PHONE_PATTERN.subn("[전화]", p_temp)
        total_masked += (email_cnt + phone_cnt)
        
        all_hashtags.extend(extract_hashtags(post))
        
    avg_length = round(total_length / posts_n, 2) if posts_n > 0 else 0.0
    
    hashtag_counts = dict(Counter(all_hashtags).most_common())
    
    return {
        "posts_n": posts_n,
        "avg_length_after_clean": avg_length,
        "hashtag_counts": hashtag_counts,
        "masked_count": total_masked
    }

posts: list[str] = [
    "오늘 #파이썬 수업 진짜 재밌었음!! @prof_kim @hong 감사 ㅎㅎ ",
    "자료: https://etl.snu.ac.kr/lec17",
    "@lee @park 팀플 어디서 모이지ㅠㅠ #DCCP2026 #팀플 카톡 ㄱㄱ",
    "<b>중요</b>: 다음 시험 범위는 1-15강. ",
    "문의는 mam3b@snu.ac.kr (010-1234-5678)로!",
    " 여러 공백과\n\n\n줄바꿈이 많은 텍스트 ",
    "ㅋㅋㅋ #파이썬 진짜 좋다 #추천 https://snu.ac.kr",
    ]

print("각 게시물에 대한 clean_post 반환값")
for i, post in enumerate(posts, 1):
    cleaned = clean_post(post)
    print(f"게시물 {i}: {repr(cleaned)}")

print("analyze_posts(posts) 반환 딕셔너")
analysis_result = analyze_posts(posts)

import json
print(json.dumps(analysis_result, ensure_ascii=False, indent=2))

각 게시물에 대한 clean_post 반환값
게시물 1: '오늘 수업 진짜 재밌었음!! 감사'
게시물 2: '자료:'
게시물 3: '팀플 어디서 모이지 카톡'
게시물 4: '중요: 다음 시험 범위는 1-15강.'
게시물 5: '문의는 [이메일] ([전화])로!'
게시물 6: '여러 공백과 줄바꿈이 많은 텍스트'
게시물 7: '진짜 좋다'
analyze_posts(posts) 반환 딕셔너
{
  "posts_n": 7,
  "avg_length_after_clean": 13.57,
  "hashtag_counts": {
    "파이썬": 2,
    "DCCP2026": 1,
    "팀플": 1,
    "추천": 1
  },
  "masked_count": 2
}


실행 결과
각 게시물에 대한 clean_post 반환값
게시물 1: '오늘 수업 진짜 재밌었음!! 감사'
게시물 2: '자료:'
게시물 3: '팀플 어디서 모이지 카톡'
게시물 4: '중요: 다음 시험 범위는 1-15강.'
게시물 5: '문의는 [이메일] ([전화])로!'
게시물 6: '여러 공백과 줄바꿈이 많은 텍스트'
게시물 7: '진짜 좋다'
analyze_posts(posts) 반환 딕셔너
{
  "posts_n": 7,
  "avg_length_after_clean": 13.57,
  "hashtag_counts": {
    "파이썬": 2,
    "DCCP2026": 1,
    "팀플": 1,
    "추천": 1
  },
  "masked_count": 2
}

re.compile()로 패턴을 미리 컴파일해두면 반복해서 사용할 때 좋다.
cleas_post에서 게시물 하나를 정제한다. 순서는 문제의 6단계와 같다.
analyze_posts에서는 게시물 전체를 분석한다. clean_post로 각 게시물을 정제한 뒤 길이를 분석하고 이메일과 전화번호 개수를 샌다.subn을 이용해 email_cnt에 이메일이 몇 개가 있었는지 저장한다. Counter을 이용해 등장 횟수를 계산해 hastag_counts에 저장한다. 이후 분석 결과를 딕셔너리 형태로 반환한다.